**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Time–Frequency Analysis II

Beyond the spectrogram: the Wigner–Ville distribution (razor resolution, haunted by cross-terms), reassignment & synchrosqueezing (sharpening the STFT after the fact), and empirical mode decomposition (letting the signal choose its own components). The modern chapter of the [uncertainty-principle](./Foundations_of_Signal_Processing_1.ipynb) story.

## 1. Pre-requisites

[Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (STFT/uncertainty), [Audio DSP](./Audio_Speech_DSP.ipynb) S1 (spectrogram fluency).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

fs = 1000
t = np.arange(0, 2, 1/fs)
# the test signal for the whole workshop: two crossing chirps + a tone burst
x = sig.chirp(t, 50, 2, 350) + sig.chirp(t, 350, 2, 50)     + np.where((t > 0.8) & (t < 1.2), np.sin(2*np.pi*420*t), 0)

---
### 🕐 Session 1 of 3 — *The Wigner–Ville Distribution* (~40 min)
**Goal:** quadratic time-frequency: perfect chirp concentration, ghost cross-terms — both demonstrated.
**Builds on:** [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S6. &nbsp; **Feeds into:** Session 2 (reassignment).

---

## 2. Correlating the Signal with Itself

💡 **Intuition.** The spectrogram windows *then* transforms — the window's blur is baked in. **Wigner–Ville** drops the window: correlate the signal with itself around each instant, $W(t, f) = \int x(t+\tau/2) x^*(t-\tau/2) e^{-j2\pi f\tau} d\tau$. For a lone linear chirp it is *perfectly* concentrated — no uncertainty smear at all (quadratic transforms don't violate the uncertainty principle; they sidestep its linear-transform premise). The price is steep: being quadratic, **every pair of components births a ghost** midway between them, oscillating — cross-terms that can dwarf the real signal.

In [2]:
def wigner_ville(x_a, n_freq=512):
    N = len(x_a)
    xa = sig.hilbert(x_a)                                 # analytic signal halves the ghosts
    W = np.zeros((n_freq, N))
    for n_i in range(N):
        tau_max = min(n_i, N-1-n_i, n_freq//2 - 1)
        tau = np.arange(-tau_max, tau_max+1)
        acf = xa[n_i + tau] * np.conj(xa[n_i - tau])
        row = np.zeros(n_freq, complex)
        row[tau % n_freq] = acf
        W[:, n_i] = np.real(np.fft.fft(row))
    return W

xa_short = x[::2][:600]                                   # decimate for speed
W = wigner_ville(xa_short)
f_stft, t_stft, S = sig.stft(x, fs=fs, nperseg=128)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].pcolormesh(t_stft, f_stft, np.abs(S), shading="auto")
axes[0].set_title("spectrogram: honest but blurred"); axes[0].set_ylim(0, 500)
axes[1].imshow(np.abs(W[:150]), aspect="auto", origin="lower",
               extent=[0, 1.2, 0, 150*fs/2/512])
axes[1].set_title("Wigner–Ville: razor lines + GHOSTS between them")
plt.tight_layout(); plt.show()
print("the ghost midway between the two chirps is not a signal — it's the quadratic cross-term")

the ghost midway between the two chirps is not a signal — it's the quadratic cross-term


/tmp/ipykernel_2986310/1466983963.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Reassignment & Synchrosqueezing* (~40 min)
**Goal:** sharpen the spectrogram by moving each energy blob to its local center of gravity.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (EMD).

---

## 3. Sharpening After the Fact

💡 **Intuition.** The spectrogram's smear has structure: within each blurry blob, the *phase* of the STFT knows where the true ridge is (the local instantaneous frequency is the phase's time-derivative). **Synchrosqueezing** reads that derivative and moves each coefficient's energy to its rightful frequency — linear-transform honesty (no cross-terms!) with nearly Wigner-grade sharpness on separated components. The catch: components must be separated by more than a window bandwidth — crossing chirps still confuse it at the crossing.

In [3]:
# synchrosqueezing-lite: reassign STFT energy along frequency by the phase derivative
f_s, t_s, Z = sig.stft(x, fs=fs, nperseg=256, noverlap=224, return_onesided=True)
eps = 1e-8
# instantaneous frequency estimate per bin: d(phase)/dt via adjacent frames
dphase = np.angle(Z[:, 1:] * np.conj(Z[:, :-1]))
hop = 256 - 224
inst_f = dphase / (2*np.pi*hop/fs)
inst_f = np.where(inst_f < 0, inst_f + fs/hop, inst_f)     # unwrap into [0, fs/hop)

S_sq = np.zeros_like(np.abs(Z[:, :-1]))
df = f_s[1] - f_s[0]
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]-1):
        mag = np.abs(Z[i, j])
        if mag < 1e-3: continue
        k = int(round(inst_f[i, j] / df))
        if 0 <= k < S_sq.shape[0]:
            S_sq[k, j] += mag

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axes[0].pcolormesh(t_s, f_s, np.abs(Z), shading="auto"); axes[0].set_title("STFT magnitude")
axes[1].pcolormesh(t_s[:-1], f_s, S_sq, shading="auto"); axes[1].set_title("synchrosqueezed: ridges snap sharp, no ghosts")
for ax in axes: ax.set_ylim(0, 500)
plt.tight_layout(); plt.show()

# quantify: ridge width (freq-axis spread) at t = 0.5 s for the rising chirp (~125 Hz)
j0 = np.argmin(np.abs(t_s - 0.5))
def width(M, j):
    col = M[(f_s > 60) & (f_s < 200), j]; col = col/ (col.sum()+eps)
    fc = f_s[(f_s > 60) & (f_s < 200)]
    mu = (col*fc).sum(); return np.sqrt((col*(fc-mu)**2).sum())
print(f"ridge width at t=0.5s:  STFT {width(np.abs(Z), j0):.1f} Hz  →  squeezed {width(S_sq, j0):.1f} Hz")

ridge width at t=0.5s:  STFT 9.1 Hz  →  squeezed 0.0 Hz


/tmp/ipykernel_2986310/680984120.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Empirical Mode Decomposition* (~35 min)
**Goal:** no basis at all: sift the signal into its own intrinsic oscillations.
**Builds on:** Session 2.

---

## 4. Let the Signal Choose

💡 **Intuition.** Fourier imposes sinusoids; wavelets impose scales. **EMD** imposes nothing: repeatedly *sift* — fit envelopes through the maxima and minima, subtract their mean — until what remains oscillates symmetrically (an *intrinsic mode function*), then peel it off and repeat. Nonlinear, adaptive, basis-free — and correspondingly fragile (mode mixing, no clean theory; ensemble-EMD patches it with noise). Powerful on nonstationary, nonlinear data; use with eyes open.

In [4]:
def emd(x_in, max_imfs=4, sift_iters=8):
    imfs, resid = [], x_in.astype(float).copy()
    for _ in range(max_imfs):
        h = resid.copy()
        for _ in range(sift_iters):
            maxima = sig.argrelextrema(h, np.greater)[0]
            minima = sig.argrelextrema(h, np.less)[0]
            if len(maxima) < 4 or len(minima) < 4: break
            from scipy.interpolate import CubicSpline
            upper = CubicSpline(maxima, h[maxima], bc_type="natural")(np.arange(len(h)))
            lower = CubicSpline(minima, h[minima], bc_type="natural")(np.arange(len(h)))
            h = h - (upper + lower)/2
        imfs.append(h); resid = resid - h
        if len(sig.argrelextrema(resid, np.greater)[0]) < 4: break
    return imfs, resid

# planted two-scale signal: fast oscillation + slow oscillation + trend — can EMD unmix it?
t2 = np.linspace(0, 1, 2000)
fast, slow, trend = np.sin(2*np.pi*60*t2), 0.8*np.sin(2*np.pi*7*t2), 1.5*t2**2
imfs, resid = emd(fast + slow + trend)

fig, axes = plt.subplots(len(imfs)+1, 1, figsize=(9, 1.2*(len(imfs)+1)), sharex=True)
for ax, imf in zip(axes, imfs): ax.plot(t2, imf, linewidth=0.6)
axes[-1].plot(t2, resid, linewidth=0.8, color="k"); axes[-1].set_title("residual (the trend)", fontsize=8)
plt.suptitle("EMD sifts: fast IMF, slow IMF, trend — no basis was specified")
plt.tight_layout(); plt.show()

print(f"IMF1 vs planted fast: |corr| = {abs(np.corrcoef(imfs[0], fast)[0,1]):.3f}")
print(f"IMF2 vs planted slow: |corr| = {abs(np.corrcoef(imfs[1], slow)[0,1]):.3f}")
print(f"residual vs trend:    |corr| = {abs(np.corrcoef(resid, trend)[0,1]):.3f}")

IMF1 vs planted fast: |corr| = 1.000
IMF2 vs planted slow: |corr| = 1.000
residual vs trend:    |corr| = 1.000


/tmp/ipykernel_2986310/3626029590.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Quadratic distributions buy razor concentration at the price of ghosts; synchrosqueezing spends the STFT's phase to sharpen without them (ridge width measured); EMD abandons bases entirely and recovers planted components (correlations verified) — with fragility as the tax on adaptivity. Choose the lens per signal, and always know what artifacts your lens invents.

---
## Where next

- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — apply all three to real sound.
- [Cyclostationary Analysis](./Cyclostationary_HOS.ipynb) — a different generalization: periodicity in the *statistics*.